In [1]:
# Step 1: Download "facades" pix2pix dataset (A|B combined images)
!wget -q -O facades.tar.gz http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz
!tar -xzf facades.tar.gz
!ls -lah facades

total 28K
drwxrwxrwx 5 8140  779 4.0K Nov 21  2016 .
drwxr-xr-x 1 root root 4.0K Mar  3 04:40 ..
drwxrwxrwx 2 8140  779 4.0K Nov 21  2016 test
drwxrwxrwx 2 8140  779  12K Nov 21  2016 train
drwxrwxrwx 2 8140  779 4.0K Nov 21  2016 val


In [2]:
import glob
print("train:", len(glob.glob("/content/facades/train/*")))
print("test :", len(glob.glob("/content/facades/test/*")))
print("val  :", len(glob.glob("/content/facades/val/*")))
print("example file:", glob.glob("/content/facades/train/*")[:1])

train: 400
test : 106
val  : 100
example file: ['/content/facades/train/370.jpg']


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
Device: cuda


In [5]:
# Step 1: Clean + download + extract + verify

import os, glob

# clean old folders/files (safe even if they don't exist)
!rm -rf /content/facades /content/facades.tar.gz

# download
!wget -O /content/facades.tar.gz http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz

# show file size (must NOT be tiny)
!ls -lh /content/facades.tar.gz

# extract
!tar -xzf /content/facades.tar.gz -C /content

# list extracted structure
!ls -lah /content/facades
!ls -lah /content/facades/train | head
!python - <<'PY'
import glob
print("train count:", len(glob.glob("/content/facades/train/*")))
print("test  count:", len(glob.glob("/content/facades/test/*")))
print("val   count:", len(glob.glob("/content/facades/val/*")))


--2026-03-03 05:00:48--  http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz
Resolving efrosgans.eecs.berkeley.edu (efrosgans.eecs.berkeley.edu)... 128.32.244.190
Connecting to efrosgans.eecs.berkeley.edu (efrosgans.eecs.berkeley.edu)|128.32.244.190|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30168306 (29M) [application/x-gzip]
Saving to: ‘/content/facades.tar.gz’

/content/facades.ta 100%[===================>]  28.77M   595KB/s    in 75s     

2026-03-03 05:02:04 (393 KB/s) - ‘/content/facades.tar.gz’ saved [30168306/30168306]

-rw-r--r-- 1 root root 29M Sep  2  2018 /content/facades.tar.gz
total 28K
drwxrwxrwx 5 8140  779 4.0K Nov 21  2016 .
drwxr-xr-x 1 root root 4.0K Mar  3 05:02 ..
drwxrwxrwx 2 8140  779 4.0K Nov 21  2016 test
drwxrwxrwx 2 8140  779  12K Nov 21  2016 train
drwxrwxrwx 2 8140  779 4.0K Nov 21  2016 val
total 21M
drwxrwxrwx 2 8140 779  12K Nov 21  2016 .
drwxrwxrwx 5 8140 779 4.0K Nov 21  2016 ..
-rwxrwxr-x 1 8140 779  54K 

In [6]:
import os, glob, random
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
from torchvision.utils import save_image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# -------- Dataset: combined AB (A|B) --------
class ABFolder(Dataset):
    def __init__(self, root, size=256):
        self.files = sorted(glob.glob(os.path.join(root, "*.jpg"))) + sorted(glob.glob(os.path.join(root, "*.png")))
        if len(self.files) == 0:
            raise ValueError(f"No images found in {root}")
        self.tf = T.Compose([
            T.Resize((size, size)),
            T.ToTensor(),
            T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
        ])
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        w, h = img.size
        w2 = w // 2
        A = img.crop((0, 0, w2, h))
        B = img.crop((w2, 0, w, h))
        return self.tf(A), self.tf(B)

# -------- Models --------
def down(in_c, out_c, norm=True):
    layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
    if norm: layers += [nn.BatchNorm2d(out_c)]
    layers += [nn.LeakyReLU(0.2, inplace=True)]
    return nn.Sequential(*layers)

def up(in_c, out_c, dropout=False):
    layers = [nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
              nn.BatchNorm2d(out_c),
              nn.ReLU(inplace=True)]
    if dropout: layers += [nn.Dropout(0.5)]
    return nn.Sequential(*layers)

class G_UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = down(3, 64, norm=False)
        self.d2 = down(64, 128)
        self.d3 = down(128, 256)
        self.d4 = down(256, 512)
        self.d5 = down(512, 512)
        self.u1 = up(512, 512, dropout=True)
        self.u2 = up(1024, 256)
        self.u3 = up(512, 128)
        self.u4 = up(256, 64)
        self.out = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh())
    def forward(self, x):
        d1 = self.d1(x); d2 = self.d2(d1); d3 = self.d3(d2); d4 = self.d4(d3); d5 = self.d5(d4)
        u1 = self.u1(d5)
        u2 = self.u2(torch.cat([u1, d4], 1))
        u3 = self.u3(torch.cat([u2, d3], 1))
        u4 = self.u4(torch.cat([u3, d2], 1))
        return self.out(torch.cat([u4, d1], 1))

class D_Patch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 1)
        )
    def forward(self, a, b):
        return self.net(torch.cat([a, b], 1))

def denorm(x): return (x*0.5)+0.5

# -------- Paths --------
TRAIN_DIR = "/content/facades/train"

# Use a subset so it runs fast
full_ds = ABFolder(TRAIN_DIR, size=256)
subset = Subset(full_ds, list(range(min(200, len(full_ds)))))
dl = DataLoader(subset, batch_size=4, shuffle=True, num_workers=2, drop_last=True)

G = G_UNet().to(DEVICE)
D = D_Patch().to(DEVICE)

optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()
lambda_l1 = 100

os.makedirs("/content/out_samples", exist_ok=True)

epochs = 1
step = 0

for ep in range(1, epochs+1):
    for a, b in dl:
        a, b = a.to(DEVICE), b.to(DEVICE)

        # D
        with torch.no_grad():
            fake_b = G(a)
        lossD = 0.5*(bce(D(a,b), torch.ones_like(D(a,b))) + bce(D(a,fake_b), torch.zeros_like(D(a,fake_b))))
        optD.zero_grad(); lossD.backward(); optD.step()

        # G
        fake_b = G(a)
        lossG = bce(D(a,fake_b), torch.ones_like(D(a,fake_b))) + lambda_l1*l1(fake_b,b)
        optG.zero_grad(); lossG.backward(); optG.step()

        step += 1
        if step % 50 == 0:
            G.eval()
            with torch.no_grad():
                a0, b0 = full_ds[random.randint(0, len(full_ds)-1)]
                a0 = a0.unsqueeze(0).to(DEVICE)
                pred = G(a0).cpu()
                grid = torch.cat([denorm(a0.cpu()), denorm(pred), denorm(b0.unsqueeze(0))], dim=0)
                save_image(grid, f"/content/out_samples/step_{step:04d}.png", nrow=3)
            G.train()
            print("Saved sample:", f"/content/out_samples/step_{step:04d}.png")

print("Done. Open /content/out_samples in Files panel.")

Device: cuda
Saved sample: /content/out_samples/step_0050.png
Done. Open /content/out_samples in Files panel.


In [7]:
import os, glob
from PIL import Image
import torch
import torchvision.transforms as T
from torchvision.utils import save_image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_DIR = "/content/facades/test"
OUT_DIR  = "/content/test_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# same preprocessing as training
tf = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

def denorm(x):  # [-1,1] -> [0,1]
    return (x * 0.5) + 0.5

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg"))) + sorted(glob.glob(os.path.join(TEST_DIR, "*.png")))
print("Test images found:", len(test_files))

G.eval()
with torch.no_grad():
    for i, f in enumerate(test_files[:20]):  # generate for first 20 test images
        img = Image.open(f).convert("RGB")
        w, h = img.size
        w2 = w // 2
        A = img.crop((0, 0, w2, h))
        B = img.crop((w2, 0, w, h))

        a = tf(A).unsqueeze(0).to(DEVICE)
        pred = G(a).cpu()

        b = tf(B).unsqueeze(0)  # on CPU is fine
        grid = torch.cat([denorm(a.cpu()), denorm(pred), denorm(b)], dim=0)
        save_image(grid, os.path.join(OUT_DIR, f"test_{i:03d}.png"), nrow=3)

print("✅ Saved outputs to:", OUT_DIR)
print("Open /content/test_outputs in the Files panel.")

Test images found: 106
✅ Saved outputs to: /content/test_outputs
Open /content/test_outputs in the Files panel.
